In [1]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score
from tscglue.models import LokyStackerV10FM
from tscglue import data_loader
import polars as pl
from sklearn.metrics import log_loss

In [3]:
dataset = "Worms"
# dataset = 'Car'
# dataset = 'HandOutlines'
#dataset = 'Trace'
#dataset = 'SwedishLeaf'
#dataset = 'Meat'
# dataset='ACSF1'
# dataset='ElectricDevices'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 19)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(181, 1, 900) (181,) (77, 1, 900) (77,)


In [4]:
seed = 2683

In [5]:
from tscglue.models import LokyStackerV10RSTSFRandom, LokyStackerV10RSTSFRandomMultiStack

m = LokyStackerV10RSTSFRandomMultiStack(random_state=seed, n_jobs=16, verbose=1, selection='best-base')

In [6]:
m.fit(X_train, y_train)

[0.00s] Starting fit, run_dir=tscglue_runs/4f2a861748bd4dc0, n_jobs=16
[4.01s] Fit+transformed multirocket_s_1675711793 features (181, 49728) (68.67 MB) dtype=float64 in 3.8719s
[12.51s] Fit+transformed hydra_s_2109509380 features (181, 7168) (9.90 MB) dtype=float64 in 8.5027s
[20.97s] Fit+transformed quant features (181, 7987) (11.03 MB) dtype=float64 in 8.4560s
[31.78s] Fit+transformed rdst_s_1973074453 features (181, 30000) (41.43 MB) dtype=float64 in 10.8167s
[39.09s] Fit+transformed rstsf-random_s_447740098 features (181, 13622) (18.81 MB) dtype=float64 in 7.3054s
[55.18s] Fit+transformed mantis features (181, 512) (0.71 MB) dtype=float64 in 16.0924s
[92.46s] Fit+transformed chronos2 features (181, 1536) (2.12 MB) dtype=float64 in 37.2773s
[94.31s] OOF acc (base) multirockethydra-bestk-p-ridgecv_s_947394068: 0.7513812154696132
[94.77s] OOF acc (base) rdst-p-ridgecv_s_198047042: 0.7237569060773481
[96.76s] OOF acc (base) fm-p-ridgecv_s_1906293317: 0.7679558011049724
[97.90s] OOF ac

/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/petelin/TS

[100.22s] OOF acc (stack) probability-nn: 0.7624309392265194
[101.86s] OOF acc (stack) probability-rf: 0.7734806629834254
[101.97s] OOF acc (stack) probability-et: 0.7734806629834254
[101.97s] Fit complete
Selected best model (best-base): fm-p-ridgecv_s_1906293317


,random_state,2683
,k_folds,10
,n_jobs,16
,verbose,1
,selection,'best-base'


In [7]:
m.classes_

array(['1', '2', '3', '4', '5'], dtype='<U1')

In [8]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

[0.00s] Starting prediction
[1.47s] Computed multirocket_s_1675711793 features (77, 49728) (29.21 MB) dtype=float64 in 1.4275s
[10.38s] Computed hydra_s_2109509380 features (77, 7168) (4.21 MB) dtype=float64 in 8.9077s
[16.30s] Computed quant features (77, 7987) (4.69 MB) dtype=float64 in 5.9254s
[19.78s] Computed rdst_s_1973074453 features (77, 30000) (17.62 MB) dtype=float64 in 3.4786s
[22.26s] Computed rstsf-random_s_447740098 features (77, 13622) (8.00 MB) dtype=float64 in 2.4762s
[37.39s] Computed mantis features (77, 512) (0.30 MB) dtype=float64 in 15.1321s
[61.71s] Computed chronos2 features (77, 1536) (0.90 MB) dtype=float64 in 24.3162s
[61.71s] Computed and saved features for prediction
[61.71s] Starting prediction with 16 workers for 50 first-level models
[62.16s] Completed all first-level model predictions
[62.26s] Starting prediction with 16 workers for 40 stacking models
[62.85s] Completed all stacking model predictions
[63.14s] Executor shutdown complete
Accuracy on Worms

In [9]:
proba = m.predict_proba(X_test)
classes = list(m.classes_)

print(f"Log-loss: {log_loss(y_test, proba, labels=classes):.4f}")
# print(f"AUC (OvR): {roc_auc_score(y_test, proba, multi_class='ovr', labels=classes):.4f}")

[0.00s] Starting prediction
[1.27s] Computed multirocket_s_1675711793 features (77, 49728) (29.21 MB) dtype=float64 in 1.2212s
[10.41s] Computed hydra_s_2109509380 features (77, 7168) (4.21 MB) dtype=float64 in 9.1427s
[16.35s] Computed quant features (77, 7987) (4.69 MB) dtype=float64 in 5.9370s
[20.12s] Computed rdst_s_1973074453 features (77, 30000) (17.62 MB) dtype=float64 in 3.7771s
[22.57s] Computed rstsf-random_s_447740098 features (77, 13622) (8.00 MB) dtype=float64 in 2.4487s
[37.58s] Computed mantis features (77, 512) (0.30 MB) dtype=float64 in 15.0079s
[61.87s] Computed chronos2 features (77, 1536) (0.90 MB) dtype=float64 in 24.2920s
[61.87s] Computed and saved features for prediction
[61.87s] Starting prediction with 16 workers for 50 first-level models
[62.85s] Completed all first-level model predictions
[62.93s] Starting prediction with 16 workers for 40 stacking models
[63.64s] Completed all stacking model predictions
[63.96s] Executor shutdown complete
Log-loss: 0.9689


In [10]:
pl.DataFrame(m.neki)

index,model,level,class,probability
i64,str,i64,str,f64
15,"""multirockethydra-bestk-p-ridge…",0,"""1""",0.220039
15,"""multirockethydra-bestk-p-ridge…",0,"""2""",0.24619
15,"""multirockethydra-bestk-p-ridge…",0,"""3""",0.233054
15,"""multirockethydra-bestk-p-ridge…",0,"""4""",0.202292
15,"""multirockethydra-bestk-p-ridge…",0,"""5""",0.098426
…,…,…,…,…
180,"""probability-et""",1,"""1""",0.003
180,"""probability-et""",1,"""2""",0.0
180,"""probability-et""",1,"""3""",0.001


In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Build probability matrix from level-0 neki predictions
df = pl.DataFrame(m.neki).filter(pl.col("level") == 0)
prob_matrix = df.pivot(on="class", index=["index", "model"], values="probability", aggregate_function='mean')
class_cols = [c for c in prob_matrix.columns if c not in ("index", "model")]
wide = prob_matrix.pivot(on="model", index="index", values=class_cols)
wide = wide.sort("index")
X_stack = wide.drop("index").to_numpy()

le = LabelEncoder()
y_enc = le.fit_transform(y_train)

stackers = {
    "RidgeCV": RidgeClassifierCV(alphas=np.logspace(-3, 3, 20)),
    "RF": RandomForestClassifier(n_estimators=200, n_jobs=-1),
    "HGBT": HistGradientBoostingClassifier(max_iter=200),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500),
}

N_REPEATS = 20
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=N_REPEATS, random_state=42)
results = {name: [] for name in stackers}

for fold_i, (tr_idx, val_idx) in enumerate(rskf.split(X_stack, y_enc)):
    rep = fold_i // 10
    for name, clf in stackers.items():
        clf_copy = clf.__class__(**clf.get_params())
        clf_copy.fit(X_stack[tr_idx], y_enc[tr_idx])
        acc = accuracy_score(y_enc[val_idx], clf_copy.predict(X_stack[val_idx]))
        results[name].append({"rep": rep, "fold": fold_i % 10, "acc": acc})

# Build test probability matrix
test_preds = m.predict_proba_per_model(X_test)
train_cols = wide.drop("index").columns

test_records = []
for model_name, proba_arr in test_preds.items():
    if model_name in m.stacking_models:
        continue
    classes_ = list(m.classes_)
    for i in range(proba_arr.shape[0]):
        for j, cls in enumerate(classes_):
            test_records.append({"index": i, "model": model_name, "level": 0, "class": str(cls), "probability": proba_arr[i, j]})

df_test = pl.DataFrame(test_records)
prob_matrix_test = df_test.pivot(on="class", index=["index", "model"], values="probability")
class_cols_test = [c for c in prob_matrix_test.columns if c not in ("index", "model")]
wide_test = prob_matrix_test.pivot(on="model", index="index", values=class_cols_test)
wide_test = wide_test.sort("index")
X_stack_test = wide_test.select(train_cols).to_numpy()
y_test_enc = le.transform(y_test)

# Test set accuracy
test_accs = {}
for name, clf in stackers.items():
    clf_final = clf.__class__(**clf.get_params())
    clf_final.fit(X_stack, y_enc)
    test_accs[name] = accuracy_score(y_test_enc, clf_final.predict(X_stack_test))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
names = list(stackers.keys())
x_positions = np.arange(len(names))

for i, name in enumerate(names):
    rdf = pl.DataFrame(results[name])
    per_rep = rdf.group_by("rep").agg(pl.col("acc").mean()).sort("rep")["acc"].to_numpy()
    # scatter all per-rep OOF accuracies
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, size=len(per_rep))
    ax.scatter(np.full_like(per_rep, i) + jitter, per_rep, alpha=0.5, s=40, zorder=3, label=f"{name} OOF reps" if i == 0 else None, color=f"C{i}")
    # mean OOF line
    ax.hlines(per_rep.mean(), i - 0.25, i + 0.25, colors=f"C{i}", linewidths=2, zorder=4)
    # test accuracy marker
    ax.scatter(i, test_accs[name], marker="*", s=200, color=f"C{i}", edgecolors="black", linewidths=0.8, zorder=5)
    print(f"{name:8s}  OOF mean={per_rep.mean():.4f} std={per_rep.std():.4f}  test={test_accs[name]:.4f}")

ax.set_xticks(x_positions)
ax.set_xticklabels(names)
ax.set_ylabel("Accuracy")
ax.set_title(f"Stacker comparison: OOF ({N_REPEATS}x10-fold) vs Test")
# custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=8, label="OOF per-rep mean"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="gray", markeredgecolor="black", markersize=15, label="Test accuracy"),
    Line2D([0], [0], color="gray", linewidth=2, label="OOF overall mean"),
]
ax.legend(handles=legend_elements, loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
